# 10 — H2a: Risk-Adjusted Alpha of the Sentiment Long-Short Portfolio

**Goal:** Test whether the sentiment-sorted long-short portfolio earns a significant
risk-adjusted alpha after controlling for the Fama-French five factors plus momentum.
This is the headline asset-pricing test (the updated Edmans, 2011 replication).

**Inputs:**
- `data/long_short_returns.parquet` (monthly Q5−Q1 returns from notebook 09)
- `data/factors.parquet` (FF5 + momentum, monthly)

**Output:** `output/table_alpha_regression.csv`

**Specification (pre-committed):**
- Dependent variable: monthly long-short (Q5−Q1) excess return
- Factors: MKT, SMB, HML, RMW, CMA (Fama & French, 2015) + MOM (Carhart, 1997)
- Newey-West (HAC) standard errors, 3 lags
- Alpha (intercept) is the coefficient of interest; annualized as α × 12

**Reference:** Edmans (2011) reported a four-factor alpha of 3.5% (2.1% industry-adjusted)
over 1984–2009 using a long portfolio. This test uses a long-short on a continuous
sentiment measure over 2013–2023 with a five-factor + momentum model.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PROCESSED = Path.home() / "thesis" / "data"
OUTPUT = Path.home() / "thesis" / "output"

ls = pd.read_parquet(DATA_PROCESSED / "long_short_returns.parquet")
factors = pd.read_parquet(DATA_PROCESSED / "factors.parquet")

print(f"Long-short series: {len(ls)} months")
print(f"Factor series: {len(factors)} months")
print()
print("Long-short columns:", list(ls.columns))
print("Factor columns:", list(factors.columns))

Long-short series: 144 months
Factor series: 156 months

Long-short columns: ['date', 'long_short', 'Q1', 'Q5']
Factor columns: ['date', 'mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'mom', 'rf', 'year', 'month']


In [2]:
# The long-short series has a 'date' column (month-end). Factors have date + year/month.
# Merge on date.
ls["date"] = pd.to_datetime(ls["date"])

reg_data = ls.merge(
    factors[["date", "mkt_rf", "smb", "hml", "rmw", "cma", "mom", "rf"]],
    on="date",
    how="inner"
)

# Restrict to the 2013-2023 headline window (per JJ's directed window) before fitting.
reg_data = reg_data[(reg_data["date"].dt.year >= 2013) & (reg_data["date"].dt.year <= 2023)].copy()

print(f"Merged regression sample: {len(reg_data)} months")
print(f"Date range: {reg_data['date'].min().date()} to {reg_data['date'].max().date()}")
print()

# The long-short is already a zero-cost portfolio (long minus short), so it's
# already an excess return — no need to subtract rf again.
# Verify no missing values in the regression variables
reg_vars = ["long_short", "mkt_rf", "smb", "hml", "rmw", "cma", "mom"]
print("Missing values per regression variable:")
print(reg_data[reg_vars].isna().sum())

Merged regression sample: 132 months
Date range: 2013-01-31 to 2023-12-31

Missing values per regression variable:
long_short    0
mkt_rf        0
smb           0
hml           0
rmw           0
cma           0
mom           0
dtype: int64


In [3]:
# Force native float64 — parquet sometimes preserves pandas nullable types
# that statsmodels cannot handle
y = reg_data["long_short"].astype("float64")

X = reg_data[["mkt_rf", "smb", "hml", "rmw", "cma", "mom"]].astype("float64")
X = sm.add_constant(X)

model = sm.OLS(y, X)
result = model.fit(cov_type="HAC", cov_kwds={"maxlags": 3})

print(result.summary())

                            OLS Regression Results                            
Dep. Variable:             long_short   R-squared:                       0.355
Model:                            OLS   Adj. R-squared:                  0.324
Method:                 Least Squares   F-statistic:                     11.77
Date:                Tue, 14 Jul 2026   Prob (F-statistic):           1.95e-10
Time:                        12:42:36   Log-Likelihood:                 384.08
No. Observations:                 132   AIC:                            -754.2
Df Residuals:                     125   BIC:                            -734.0
Df Model:                           6                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0012      0.001      1.373      0.1

In [4]:
import sys
print(f"Python: {sys.version.split()[0]}")
print(f"Executable: {sys.executable}")

Python: 3.11.15
Executable: /opt/anaconda3/envs/thesis/bin/python


In [5]:
# Extract key results into a clean table
alpha_table = pd.DataFrame({
    "coefficient": result.params,
    "std_error": result.bse,
    "z_stat": result.tvalues,
    "p_value": result.pvalues,
})
alpha_table.loc["const", "annualized_alpha"] = result.params["const"] * 12
print(alpha_table.round(4))

alpha_table.round(4).to_csv(OUTPUT / "table_alpha_regression.csv")
print(f"\nSaved to {OUTPUT / 'table_alpha_regression.csv'}")

        coefficient  std_error  z_stat  p_value  annualized_alpha
const        0.0012     0.0009  1.3729   0.1698            0.0143
mkt_rf       0.0540     0.0343  1.5748   0.1153               NaN
smb         -0.2251     0.0627 -3.5919   0.0003               NaN
hml         -0.0917     0.0487 -1.8811   0.0600               NaN
rmw         -0.1639     0.0777 -2.1093   0.0349               NaN
cma         -0.1959     0.0758 -2.5864   0.0097               NaN
mom         -0.0793     0.0455 -1.7421   0.0815               NaN

Saved to /Users/<wrds-username>/thesis/output/table_alpha_regression.csv


In [6]:
# ADDED 2026-07-14 after external review (post-hoc, not pre-committed):
# (a) Newey-West lag-window sensitivity of the headline equal-weighted alpha
#     (3 lags is the spec; 6 and 12 reported for transparency);
# (b) six-factor alpha of the value-weighted spread built in notebook 09.
rows = []
for lags in (3, 6, 12):
    res_l = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
    rows.append({"series": "EW long-short", "nw_lags": lags,
                 "alpha_pct_yr": res_l.params["const"] * 12 * 100,
                 "t_stat": res_l.tvalues["const"], "n_months": int(res_l.nobs)})
    print(f"EW alpha, NW({lags:>2}): {res_l.params['const']*12*100:6.2f}%/yr, t = {res_l.tvalues['const']:.2f}")

ls_vw = pd.read_parquet(DATA_PROCESSED / "long_short_returns_vw.parquet")
ls_vw["date"] = pd.to_datetime(ls_vw["date"])
reg_vw = ls_vw.merge(factors[["date", "mkt_rf", "smb", "hml", "rmw", "cma", "mom"]],
                     on="date", how="inner")
reg_vw = reg_vw[(reg_vw["date"].dt.year >= 2013) &
                (reg_vw["date"].dt.year <= 2023)].dropna(subset=["long_short_vw"]).copy()
y_vw = reg_vw["long_short_vw"].astype("float64")
X_vw = sm.add_constant(reg_vw[["mkt_rf", "smb", "hml", "rmw", "cma", "mom"]].astype("float64"))
print()
for lags in (3, 6, 12):
    res_v = sm.OLS(y_vw, X_vw).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
    rows.append({"series": "VW long-short", "nw_lags": lags,
                 "alpha_pct_yr": res_v.params["const"] * 12 * 100,
                 "t_stat": res_v.tvalues["const"], "n_months": int(res_v.nobs)})
    print(f"VW alpha, NW({lags:>2}): {res_v.params['const']*12*100:6.2f}%/yr, t = {res_v.tvalues['const']:.2f}")

res_v3 = sm.OLS(y_vw, X_vw).fit(cov_type="HAC", cov_kwds={"maxlags": 3})
print("\nVW six-factor loadings (NW 3):")
print(pd.DataFrame({"coef": res_v3.params, "t": res_v3.tvalues}).round(3))

pd.DataFrame(rows).round(4).to_csv(OUTPUT / "table_alpha_sensitivity.csv", index=False)
print(f"\nSaved {OUTPUT / 'table_alpha_sensitivity.csv'}")

EW alpha, NW( 3):   1.43%/yr, t = 1.37
EW alpha, NW( 6):   1.43%/yr, t = 1.43
EW alpha, NW(12):   1.43%/yr, t = 1.32

VW alpha, NW( 3):   3.53%/yr, t = 1.76
VW alpha, NW( 6):   3.53%/yr, t = 1.82
VW alpha, NW(12):   3.53%/yr, t = 1.91

VW six-factor loadings (NW 3):
         coef      t
const   0.003  1.764
mkt_rf  0.129  2.232
smb    -0.317 -3.238
hml    -0.231 -2.990
rmw    -0.086 -0.641
cma    -0.220 -1.714
mom    -0.090 -1.224

Saved /Users/<wrds-username>/thesis/output/table_alpha_sensitivity.csv
